In [ ]:
# docker file used for this notebook for pytorch 2.0
# docker pull cnstark/pytorch:2.0.1-py3.10.11-cuda11.8.0-ubuntu22.04

In [ ]:
import torch
print(torch.__version__)


# Семинар torch.compile 

Cеминар основан на документации [1] и ноутбуке [2].<br>

[1] https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html <br>
[2] https://colab.research.google.com/github/PyTorchKorea/tutorials-kr/blob/master/docs/_downloads/96ad88eb476f41a5403dcdade086afb8/torch_compile_tutorial.ipynb <br>

``torch.compile`` компилятор для ускорения кода на PyTorch. <br>
Это JIT-компилятор с использованием оптимизированных кернелей,
который требует минимального изменения кода. <br>

В этом семинаре мы рассмотрим базовое использование ``torch.compile``, а так же сравним его с [TorchScript](https://pytorch.org/docs/stable/jit.html) и
[FX Tracing](https://pytorch.org/docs/stable/fx.html#torch.fx.symbolic_trace).

**Содержание**

- Базовый пример
- Ускорение с помощью ``torch.compile``
- Сравнение с TorchScript и FX Tracing
- Заключение

**Required pip Dependencies**

- ``torch >= 2.0``
- ``torchvision``
- ``numpy``
- ``scipy``
- ``tabulate``

docker file used for this notebook for pytorch 2.0: <br>
``docker pull cnstark/pytorch:2.0.1-py3.10.11-cuda11.8.0-ubuntu22.04``





ПРИМЕЧАНИЕ. Результаты зависят от версии GPU, что бы воспроизвести результаты команда ``torch.compile`` рекомендует использование NVIDIA (H100, A100 или V100). 

## Базовый пример

Произвольные функции Python можно оптимизировать, передав вызываемый объект в
``torch.compile``. Затем мы можем вызвать возвращенную оптимизированную
функцию вместо исходной.



In [ ]:
import torch

def foo(x, y):
    a = torch.sin(x)
    b = torch.cos(y)
    return a + b

opt_foo = torch.compile(foo)

print(opt_foo(torch.randn(10, 10), torch.randn(10, 10)))

Так же можно использовать декоратор

In [ ]:
@torch.compile
def opt_foo2(x, y):
    a = torch.sin(x)
    b = torch.cos(y)
    return a + b
print(opt_foo2(torch.randn(10, 10), torch.randn(10, 10)))

Мы можем оптимизировать целый модуль ``torch.nn.Module``.



In [ ]:
class MyModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = torch.nn.Linear(100, 10)

    def forward(self, x):
        return torch.nn.functional.relu(self.lin(x))

mod = MyModule()

opt_mod = torch.compile(mod)
print(opt_mod(torch.randn(10, 100)))


## Ускорение для инференса

Давайте теперь посмотрим, как использование ``torch.compile`` может ускорить <br>
модели. Мы сравним стандартный eager режим и ``torch.compile`` для инфиренса и для обучения DenseNet на случайных данных.

Но прежде чем мы начнем, нам нужно определить некоторые служебные функции.


In [ ]:
# Returns the result of running `fn()` and the time it took for `fn()` to run,
# in seconds. We use CUDA events and synchronization for the most accurate
# measurements.
def timed(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, start.elapsed_time(end) / 1000

# Generates random input and targets data for the model, where `b` is
# batch size.
def generate_data(b):
    return (
        torch.randn(b, 3, 128, 128).to(torch.float32).to('cuda:1'),
        torch.randint(1000, (b,)).to('cuda:1'),
    )

from torchvision.models import densenet121
def init_model():
    return densenet121().to(torch.float32).to('cuda:1')

Первым делом мы сравним инференс, но прежде посмотрим сколько времени занимает компиляция.

В  ``torch.compile`` мы используем некотрые параметры для переменной ``mode``, мы обсудим их позже.


In [ ]:
def evaluate(mod, inp):
    return mod(inp)

model = init_model()

# Reset since we are using a different model.
import torch._dynamo
torch._dynamo.reset()

evaluate_opt = torch.compile(evaluate)
evaluate_opt1 = torch.compile(evaluate, mode="reduce-overhead")
evaluate_opt2 = torch.compile(evaluate, mode="max-autotune")


inp = generate_data(16)[0]

# One Iteration for each function
print("standard:", timed(lambda: evaluate_opt(model, inp))[1])
print("reduce-overhead:", timed(lambda: evaluate_opt1(model, inp))[1])
print("max-autotune:", timed(lambda: evaluate_opt2(model, inp))[1])
print("original", timed(lambda: evaluate(model, inp))[1])

In [ ]:
# One Iteration for each function 
print("standard:", timed(lambda: evaluate_opt(model, inp))[1])
print("reduce-overhead:", timed(lambda: evaluate_opt1(model, inp))[1])
print("max-autotune:", timed(lambda: evaluate_opt2(model, inp))[1])
print("original", timed(lambda: evaluate(model, inp))[1])

Обратите внимание, что выполнение ``torch.compile`` занимает намного больше времени <br>
по сравнению с ``eager mode``. Это потому, что ``torch.compile`` занимается компиляцией и  <br>
оптимизирует ядра по мере ее выполнения.  Здесь мы запускаем нашу модель один раз. <br>

Если мы будем запускать нашу модель несколько раз, то после компиляции, которая происходит на первом этапе мы увидим ускорение. <br>

In [ ]:
@torch.no_grad()
def evaluate(mod, inp):
    return mod(inp)

model = init_model()

# Reset since we are using a different model.
import torch._dynamo
torch._dynamo.reset()

inp = generate_data(16)[0]

evaluate_opt = torch.compile(evaluate, mode="max-autotune")
print("max-autotune:", timed(lambda: evaluate_opt(model, inp))[1])


In [ ]:
print("max-autotune:", timed(lambda: evaluate_opt(model, inp))[1])

In [ ]:
model = init_model().eval()

In [ ]:
N_ITERS = 10

eager_times = []
compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    _, eager_time = timed(lambda: evaluate(model, inp))
    eager_times.append(eager_time)
    print(f"eager eval time {i}: {eager_time}")

print("~" * 10)

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    _, compile_time = timed(lambda: evaluate_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile eval time {i}: {compile_time}")
print("~" * 10)

import numpy as np
eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
print(f"(eval) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

In [ ]:
N_ITERS = 10

model = init_model()
opt = torch.optim.Adam(model.parameters())

def train(mod, data):
    opt.zero_grad(True)
    pred = mod(data[0])
    loss = torch.nn.CrossEntropyLoss()(pred, data[1])
    loss.backward()
    opt.step()

eager_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, eager_time = timed(lambda: train(model, inp))
    eager_times.append(eager_time)
    print(f"eager train time {i}: {eager_time}")
print("~" * 10)

model = init_model()
opt = torch.optim.Adam(model.parameters())
train_opt = torch.compile(train, mode="max-autotune")

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, compile_time = timed(lambda: train_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile train time {i}: {compile_time}")
print("~" * 10)

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
print(f"(train) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

In [ ]:
compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, compile_time = timed(lambda: train_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile train time {i}: {compile_time}")
print("~" * 10)

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
print(f"(train) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x")
print("~" * 10)

Опять, мы видим, что ``torch.compile`` первая итерация занимает больше времени.
итерации, так как она должна скомпилировать модель, но на последующих итерациях мы видим
значительное ускорение по сравнению с нетерпеливым.


## Cравнение с TorchScript и FX Tracing

Мы видели что ``torch.compile`` может ускорить вычисления. <br>


Прежде всего, преимущество ``torch.compile`` заключается в его способности работать
почти с произвольным кодом на Python с его минимальными изменениями. <br>

А так же ``torch.compile`` может работать с data-dependent control flow ``if x.sum() < 0:``.



In [ ]:
def f1(x):
    if x < 10:
        return torch.tensor(0.)
    return x

# Test that `fn1` and `fn2` return the same result, given
# the same arguments `args`. Typically, `fn1` will be an eager function
# while `fn2` will be a compiled function (torch.compile, TorchScript, or FX graph).
def test_fns(fn1, fn2, args):
    out1 = fn1(*args)
    out2 = fn2(*args)
    return print(out1, out2)

inp1 = torch.tensor(5)
inp2 = torch.tensor(15)

TorchScript tracing ``f1`` привдет к неправильным результатам, так  ак зафиксируется детерменированный проход по данным.



In [ ]:
traced_f1 = torch.jit.trace(f1, (inp1))
f1(inp1), traced_f1(inp1)

In [ ]:
f1(inp2), traced_f1(inp2)

```
TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.sum() < 0:
```

FX tracing ``f1`` приводит к ошибке из-за присутствия поток управления, зависящий от данных.



In [ ]:
import traceback as tb
try:
    torch.fx.symbolic_trace(f1)
except:
    tb.print_exc()

Если мы предоставим значение для ``x`` при попытке  FX  trace ``f1``, тогда
мы сталкиваемся с той же проблемой, что и при TorchScript, из за зависимости графа от данных.

In [ ]:
fx_f1 = torch.fx.symbolic_trace(f1, concrete_args={"x": inp1})
fx_f1(inp1), fx_f1(inp2)

В это время ``torch.compile`` корректно обрабатывает динамический control-flow.



In [ ]:
# Reset since we are using a different mode.
torch._dynamo.reset()

compile_f1 = torch.compile(f1)
compile_f1(inp1), compile_f1(inp2)

TorchScript scripting может работать в этой ситуации, но нам нужно будет адаптировать код и убедиться что у нас используется статическое типизирование данных.



In [ ]:
script_f1 = torch.jit.script(f1)
script_f1(inp1), script_f1(inp2)

In [ ]:
def f2(x, y):
    return x + y

inp1 = torch.randn(5, 5)
inp2 = 3

script_f2 = torch.jit.script(f2)
try:
    script_f2(inp1, inp2)
except:
    tb.print_exc()

In [ ]:
inp2 = torch.tensor(3.0)


script_f2 = torch.jit.script(f2)
try:
    script_f2(inp1, inp2)
except:
    tb.print_exc()

Другое преимущество ``torch.compile``  это возможность использлвать больше функций.


In [ ]:
import scipy

def f3(x):
    x = x * 2
    x = scipy.fft.dct(x.numpy())
    x = torch.from_numpy(x)
    x = x * 2
    return x

TorchScript обрабатывает результаты вызовов функций, отличных от PyTorch.
как константы, и поэтому результаты могут быть ошибочными.

In [ ]:
import numpy as np

In [ ]:
inp1 = torch.randn(5, 5)
inp2 = torch.randn(5, 5)
traced_f3 = torch.jit.trace(f3, (inp1,))

In [ ]:
f3(inp2)

In [ ]:
traced_f3(inp2)

``torch.compile`` 


In [ ]:
inp2 = torch.randn(2, 2)
compile_f3 = torch.compile(f3)
compile_f3(inp1), compile_f3(inp2), np.allclose(f3(inp2), compile_f3(inp2))

## TorchDynamo и FX Graphs

Одна из важных компонент ``torch.compile`` это TorchDynamo.
TorchDynamo отвечает за JIT компиляцию кода на Python в
[FX graphs](https://pytorch.org/docs/stable/fx.html#torch.fx.Graph), который далее может быть оптимизирован. TorchDynamo выделяет FX графы анализруя Питоновский байткод во время выполнения, а так же отслеживает вызовы к функциям PyTorch.

TorchInductor, другая компонента ``torch.compile``,
конвертирует FX граф в оптимизированные кернели, а
TorchDynamo позволяет использовать различные бекнеды.  <br>


Давайте создадим свой бекенд, который выводит FX граф и не оптимизированный forward (что-то типа принта, но вызывывается он TorchDynamo). 

In [ ]:
from typing import List

def custom_backend(gm: torch.fx.GraphModule, example_inputs: List[torch.Tensor]):
    print("custom backend called with FX graph:")
    gm.graph.print_tabular()
    return gm.forward

# Reset since we are using a different backend.
torch._dynamo.reset()

opt_model = torch.compile(init_model(), backend=custom_backend)
opt_model(generate_data(16)[0])


Используя наш собственный бэкэнд, мы теперь можем увидеть, как TorchDynamo работает с ситуацией, когда есть зависимость от данных. Рассмотрим:
``if b.sum() < 0``. 



In [ ]:
def bar(a, b):
    x = a / (torch.abs(a) + 1)
    if b.sum() < 0:
        b = b * -1
    return x * b

opt_bar = torch.compile(bar, backend=custom_backend)
inp1 = torch.randn(10)
inp2 = torch.randn(10)
opt_bar(inp1, inp2)
opt_bar(inp1, -inp2)

Мы видим что TorchDynamo разбил наш код на три части, которые соотвествуют:


1. ``x = a / (torch.abs(a) + 1)``
2. ``b = b * -1; return x * b``
3. ``return x * b``

Когда TorchDynamo встречает не поддерживаемые в функции, например зависящие от входных данных, он  разбивает код на части, дает передает все в стандатный Python интерпритатор, а потом возвращается к графу.

Давайте на примере разберемся, как TorchDynamo пройдет через ``<``.
Если ``b.sum() < 0``, то TorchDynamo запустит граф 1, даст
Python определить результат условного выражения, а затем запустит
граф 2. С другой стороны, если ``не b.sum() < 0``, то TorchDynamo
запустил бы граф 1, позволил Python определить результат условного выражения, затем
запустил график 3. <br>


Это подчеркивает основное различие между TorchDynamo и предыдущим  компиляторами для PyTorch.

Предыдущие решения либо упадут с ошибков, либо не правильно скомпилируются и ничего не произойдет.
TorchDynamo, с другой стороны, просто разобьет граф вычислений на части.